# v0.19 Audit-Corrected Research & Forward Microstructure
This notebook reproduces the v0.19 audit and public-data microstructure smoke test. It **does not** enable real-money trading or require exchange API secrets.

Scientific contract: audit results on spent/retrospective samples cannot promote a strategy. Prospective microstructure snapshots are evidence collection only.

In [ ]:
!rm -rf modular-crypto-trading-bot
!git clone https://github.com/parsa314/modular-crypto-trading-bot.git
%cd modular-crypto-trading-bot
!git fetch --all
!git checkout v19-audit-corrected-forward-microstructure


In [ ]:
!python -m pip install -q --upgrade pip
!pip install -q -e '.[dev]'


## 1) Deterministic audit tests

In [ ]:
!pytest -q tests/test_audit_v19.py tests/test_forward_microstructure_v19.py tests/test_cost_regime_v18.py


## 2) Audit-corrected re-analysis (non-promotional)

In [ ]:
!python scripts/run_v19_audit.py --bars 1200 --output artifacts/v19/v19_audit.json --markdown artifacts/v19/V19_AUDIT_RESULTS.md


In [ ]:
import json
from pathlib import Path
audit = json.loads(Path('artifacts/v19/v19_audit.json').read_text())
print('manifest:', audit['manifest_sha256'])
print('cost audit status:', audit['cost_audit']['evidence_status'])
print('regime audit status:', audit['regime_audit']['evidence_status'])
print('live authorized:', audit['live_execution_authorized'])


## 3) Prospective multi-venue microstructure snapshot

In [ ]:
!python scripts/collect_v19_forward_microstructure.py --output artifacts/v19/forward_microstructure_snapshot.json


In [ ]:
snap = json.loads(Path('artifacts/v19/forward_microstructure_snapshot.json').read_text())
print('snapshot:', snap['snapshot_sha256'])
print('authorized symbols:', snap['authorized_symbol_count'])
print('signal authorized:', snap['signal_authorized'])
for row in snap['symbols']:
    print(row['symbol'], row['status'], row.get('accepted_venues'), row.get('mean_trade_imbalance'))


## Interpretation
- v0.18-A uncertainty calibration is re-audited with expanding out-of-fold residuals.
- terminal closing costs are charged.
- v0.18-B is reconstructed using the v0.11 cross-sectional top-quartile portfolio geometry and market-level regime definition.
- the corrected retrospective results are **not fresh alpha evidence**.
- the scheduled v0.19 collector starts the fresh prospective evidence stream.